# LeetCode 1706 — Where Will the Ball Fall bằng Python

Link đề: https://leetcode.com/problems/where-will-the-ball-fall/ · Độ khó: **Medium** · Chủ đề: Array, Matrix, Simulation, Dynamic Programming, Depth-First Search

Skeleton LeetCode:

```python
class Solution:
    def findBall(self, grid: List[List[int]]) -> List[int]:
        ...
```

Notebook này đi từ cách **mô phỏng thẳng theo đề** (kiểm tra 4 trường hợp kẹt) tới quy tắc gọn một dòng `nj = j + grid[i][j]`, rồi tới **DP từ đáy lên** trả lời mọi cột xuất phát cùng lúc. Cuối notebook có bộ test so sánh với một **mô hình tam giác độc lập** (tự dựng hình học từng ô) để chắc chắn các cách đều đúng.

## 1. Mục tiêu học

1. Đọc một đề **mô phỏng trên lưới** và chuyển hình vẽ (tấm ván chéo, chữ "V", tường) thành các **điều kiện chỉ số** chính xác.
2. Viết được lời giải mô phỏng từng quả bóng với **4 trường hợp kẹt** rồi rút gọn về **một quy tắc duy nhất**: "ô bên cạnh theo hướng ván phải có ván cùng hướng".
3. Viết cùng thuật toán theo 3 phong cách: vòng lặp, **đệ quy DFS**, và **DP từ đáy lên** (bảng `exit[i][j]`), hiểu khi nào mỗi phong cách có lợi.
4. Nhận ra bẫy riêng của Python: **chỉ số âm không báo lỗi** (`grid[i][-1]` lấy cột cuối) nên quên kiểm tra tường trái là lỗi *im lặng*.
5. Kiểm thử bằng một mô hình **khác hẳn về cách nghĩ** (chia mỗi ô thành 2 tam giác) trên nhiều lưới ngẫu nhiên có seed cố định.

## 2. Đề bài & ví dụ

Có một chiếc hộp hình lưới `grid` kích thước `m × n`, mở ở **mặt trên** và **mặt dưới**. Mỗi ô có một **tấm ván chéo** nối hai góc đối diện:

- `grid[i][j] = 1`: ván nối **góc trên-trái → góc dưới-phải** (hình `\`), đẩy bóng sang **phải**.
- `grid[i][j] = -1`: ván nối **góc trên-phải → góc dưới-trái** (hình `/`), đẩy bóng sang **trái**.

Ta thả `n` quả bóng, mỗi quả ở đỉnh một cột. Một quả bóng hoặc **kẹt** trong hộp, hoặc **rơi ra** ở đáy. Bóng kẹt khi:
- chạm vào hình chữ **"V"** tạo bởi hai tấm ván cạnh nhau (`\` rồi `/`), hoặc
- tấm ván đẩy bóng vào **tường** trái/phải của hộp.

Trả về mảng `answer` độ dài `n`: `answer[i]` là **cột ở đáy** mà quả bóng thả từ cột `i` rơi ra, hoặc `-1` nếu nó bị kẹt.

**Ví dụ 1:**
`grid = [[1,1,1,-1,-1],[1,1,1,-1,-1],[-1,-1,-1,1,1],[1,1,1,1,-1],[-1,-1,-1,-1,-1]]` → `[1,-1,-1,-1,-1]`.
Bóng 0 rơi ra ở cột 1; bóng 1 kẹt giữa cột 2–3 ở hàng 1; bóng 2 và 3 kẹt ở hàng 0 (giữa cột 2–3); bóng 4 kẹt ở hàng 1.

**Ví dụ 2:** `grid = [[-1]]` → `[-1]` (ván đẩy bóng vào tường trái).

**Ví dụ 3:** `grid = [[1,1,1,1,1,1],[-1,-1,-1,-1,-1,-1],[1,1,1,1,1,1],[-1,-1,-1,-1,-1,-1]]` → `[0,1,2,3,4,-1]`.

**Ràng buộc:**
- `m == grid.length`, `n == grid[i].length`
- `1 ≤ m, n ≤ 100`
- `grid[i][j]` là `1` hoặc `-1`.

In [1]:
import random
import time
from typing import List

examples = [
    ([[1, 1, 1, -1, -1], [1, 1, 1, -1, -1], [-1, -1, -1, 1, 1], [1, 1, 1, 1, -1], [-1, -1, -1, -1, -1]],
     [1, -1, -1, -1, -1]),
    ([[-1]], [-1]),
    ([[1, 1, 1, 1, 1, 1], [-1, -1, -1, -1, -1, -1], [1, 1, 1, 1, 1, 1], [-1, -1, -1, -1, -1, -1]],
     [0, 1, 2, 3, 4, -1]),
]

def show(grid):
    """In lưới bằng ký hiệu \\ và / cho dễ hình dung."""
    for row in grid:
        print(" ".join("\\" if v == 1 else "/" for v in row))

for g, want in examples:
    show(g)
    print("-> kỳ vọng", want)
    print()

\ \ \ / /
\ \ \ / /
/ / / \ \
\ \ \ \ /
/ / / / /
-> kỳ vọng [1, -1, -1, -1, -1]

/
-> kỳ vọng [-1]

\ \ \ \ \ \
/ / / / / /
\ \ \ \ \ \
/ / / / / /
-> kỳ vọng [0, 1, 2, 3, 4, -1]



## 3. Kiến thức nền

### 3.1 Lưới 2 chiều trong Python: `list` của `list`

`grid[i][j]` là hàng `i` (từ trên xuống), cột `j` (từ trái sang). Số hàng `m = len(grid)`, số cột `n = len(grid[0])`. Bóng luôn đi **xuống** một hàng sau mỗi ô, nên chỉ số hàng `i` chạy `0 → m − 1`; chỉ số cột `j` thay đổi ±1.

In [2]:
grid = [[1, 1, -1],
        [-1, 1, 1]]
m, n = len(grid), len(grid[0])
print("m =", m, "n =", n)
print("grid[0][2] =", grid[0][2], "| grid[1][0] =", grid[1][0])
for i in range(m):
    for j in range(n):
        print(f"({i},{j})={grid[i][j]:>2}", end="  ")
    print()

m = 2 n = 3
grid[0][2] = -1 | grid[1][0] = -1
(0,0)= 1  (0,1)= 1  (0,2)=-1  
(1,0)=-1  (1,1)= 1  (1,2)= 1  


### 3.2 Bẫy chỉ số âm

Trong nhiều ngôn ngữ, `a[-1]` gây lỗi. Trong Python, `a[-1]` là **phần tử cuối**. Vì vậy nếu bóng ở cột `0` và ta vô tình đọc `grid[i][j - 1]`, Python **không báo lỗi** mà lặng lẽ đọc cột cuối cùng — một nguồn bug rất khó thấy. Ngược lại `a[len(a)]` thì có báo `IndexError`.

In [3]:
row = [1, -1, 1, 1]
j = 0
print("row[j - 1] khi j = 0 ->", row[j - 1], "(đây là row[3], KHÔNG phải tường!)")
try:
    print(row[len(row)])
except IndexError as e:
    print("row[len(row)] -> IndexError:", e)

row[j - 1] khi j = 0 -> 1 (đây là row[3], KHÔNG phải tường!)
row[len(row)] -> IndexError: list index out of range


### 3.3 Hướng di chuyển = giá trị ô

Một mẹo rất gọn: giá trị ván **chính là độ dời cột**. Ván `1` đẩy sang phải (`j + 1`), ván `-1` đẩy sang trái (`j − 1`). Vậy cột kế tiếp luôn là `nj = j + grid[i][j]`. Không cần `if` để chọn hướng.

In [4]:
for v in (1, -1):
    j = 3
    print(f"ván {v:>2}: từ cột {j} sang cột {j + v}")

ván  1: từ cột 3 sang cột 4
ván -1: từ cột 3 sang cột 2


### 3.4 Kiểm tra nằm trong khoảng: `0 <= x < n`

Python cho phép so sánh dây chuyền `0 <= nj < n`, tương đương `0 <= nj and nj < n`. Đây là cách kiểm tra "chưa đụng tường".

### 3.5 Hàm đệ quy lồng trong hàm

Để viết DFS, ta định nghĩa `dfs(i, j)` **bên trong** `findBall` để dùng được `grid`, `m`, `n` mà không cần truyền tham số. Độ sâu đệ quy tối đa là `m ≤ 100`, rất an toàn so với giới hạn mặc định (~1000).

In [5]:
def demo_outer(n):
    hits = []
    def inside(x):          # dùng được n và hits của hàm ngoài
        if 0 <= x < n:
            hits.append(x)
            return True
        return False
    return [inside(x) for x in (-1, 0, n - 1, n)], hits

print(demo_outer(4))

([False, True, True, False], [0, 3])


### 3.6 List comprehension và `zip`

`[f(j) for j in range(n)]` tạo mảng đáp án ngắn gọn. `all(a == b for a, b in zip(x, y))` hữu ích khi so sánh hai kết quả trong phần test.

In [6]:
print([j * j for j in range(5)])
x, y = [1, -1, 2], [1, -1, 2]
print("giống nhau?", all(a == b for a, b in zip(x, y)) and len(x) == len(y))

[0, 1, 4, 9, 16]
giống nhau? True


## 4. Ý tưởng nền tảng / trực giác

Hãy nhìn **một ô** và tưởng tượng bóng rơi vào từ cạnh trên:

- Ô `\` (giá trị `1`): ván dốc xuống bên phải, bóng lăn sang **ô bên phải cùng hàng**. Nhưng nó chỉ đi tiếp xuống được nếu ô bên phải cũng là `\` — khi đó hai ván song song tạo thành một "máng" chéo, bóng trượt xuống góc dưới-phải và rơi vào **hàng dưới, cột `j + 1`**.
- Nếu ô bên phải là `/`, hai ván `\/` tạo thành chữ **V**: bóng nằm ở đáy chữ V, **kẹt**.
- Nếu không có ô bên phải (bóng ở cột cuối), ván đẩy bóng vào **tường phải**: **kẹt**.

Đối xứng cho ô `/` (giá trị `-1`) với ô bên trái.

**Tính tay ví dụ 3** (4 hàng, 6 cột), bóng thả ở cột 0:

| Hàng | Cột hiện tại | Ván | Ô cạnh (theo hướng ván) | Kết quả |
|---|---|---|---|---|
| 0 | 0 | `\` (1) | cột 1 là `\` | xuống cột 1 |
| 1 | 1 | `/` (−1) | cột 0 là `/` | xuống cột 0 |
| 2 | 0 | `\` (1) | cột 1 là `\` | xuống cột 1 |
| 3 | 1 | `/` (−1) | cột 0 là `/` | xuống cột 0 |
| — | **0** | | | rơi ra cột 0 |

Bóng thả ở cột 5: hàng 0, ván `\` đẩy sang phải nhưng cột 6 không tồn tại ⇒ **tường phải**, kẹt ⇒ `-1`. Khớp đáp án `[0,1,2,3,4,-1]`.

Quan sát quan trọng: **mỗi quả bóng độc lập**, và tại mỗi hàng bóng chỉ cần xem **2 ô**: ô đang đứng và ô cạnh nó theo hướng ván.

In [7]:
g3 = examples[2][0]
j = 0
for i in range(len(g3)):
    v = g3[i][j]
    nj = j + v
    print(f"hàng {i}: cột {j}, ván {v:>2}, ô cạnh cột {nj} = {g3[i][nj] if 0 <= nj < len(g3[0]) else 'tường'}")
    j = nj
print("rơi ra cột", j)

hàng 0: cột 0, ván  1, ô cạnh cột 1 = 1
hàng 1: cột 1, ván -1, ô cạnh cột 0 = -1
hàng 2: cột 0, ván  1, ô cạnh cột 1 = 1
hàng 3: cột 1, ván -1, ô cạnh cột 0 = -1
rơi ra cột 0


## 5. Cách 1 — Brute Force / baseline: mô phỏng theo đúng 4 trường hợp của đề

**Ý tưởng:** làm đúng như đề mô tả. Với mỗi quả bóng, đi từng hàng; ở mỗi hàng xét ván:

- Ván `1` (`\`): kẹt nếu **(a)** đang ở cột cuối (tường phải) hoặc **(b)** ô bên phải là `-1` (chữ V). Ngược lại sang cột `j + 1`.
- Ván `-1` (`/`): kẹt nếu **(c)** đang ở cột đầu (tường trái) hoặc **(d)** ô bên trái là `1` (chữ V). Ngược lại sang cột `j − 1`.

Thứ tự kiểm tra quan trọng: **kiểm tra tường trước**, rồi mới đọc ô cạnh (tránh đọc ngoài mảng hoặc chỉ số âm).

In [8]:
def find_ball_brute(grid):
    m, n = len(grid), len(grid[0])
    ans = []
    for start in range(n):
        j = start
        stuck = False
        for i in range(m):
            if grid[i][j] == 1:                 # ván "\" đẩy sang phải
                if j == n - 1:                  # (a) tường phải
                    stuck = True; break
                if grid[i][j + 1] == -1:        # (b) chữ V: "\/"
                    stuck = True; break
                j += 1
            else:                               # ván "/" đẩy sang trái
                if j == 0:                      # (c) tường trái
                    stuck = True; break
                if grid[i][j - 1] == 1:         # (d) chữ V: "\/"
                    stuck = True; break
                j -= 1
        ans.append(-1 if stuck else j)
    return ans

for g, want in examples:
    got = find_ball_brute(g)
    print(got, "| kỳ vọng", want, "| đúng" if got == want else "| SAI")

[1, -1, -1, -1, -1] | kỳ vọng [1, -1, -1, -1, -1] | đúng
[-1] | kỳ vọng [-1] | đúng
[0, 1, 2, 3, 4, -1] | kỳ vọng [0, 1, 2, 3, 4, -1] | đúng


**Độ phức tạp:** mỗi bóng đi tối đa `m` hàng, mỗi hàng `O(1)` ⇒ Time `O(m·n)`, Space `O(1)` ngoài mảng đáp án.

**Nút thắt:** thật ra với ràng buộc `m, n ≤ 100` thì cách này đã đủ nhanh (tối đa 10⁴ bước). "Nút thắt" ở đây là **độ dài và độ rối của code**: 4 nhánh `if` tách rời, dễ viết nhầm dấu, nhầm `j + 1` / `j − 1`, hoặc quên một trường hợp. Ngoài ra nếu đề hỏi **rất nhiều truy vấn** (thả bóng ở hàng bất kỳ, nhiều lần), mô phỏng lại từ đầu mỗi lần sẽ lặp công. Hai điểm đó dẫn tới các cách sau.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 — gộp 4 trường hợp thành 1.** Đặt `v = grid[i][j]` và `nj = j + v` (cột mà ván đẩy bóng tới).

- Nếu `nj` nằm **ngoài** `[0, n − 1]` ⇒ bóng bị đẩy vào tường ⇒ kẹt. (Gộp (a) và (c).)
- Nếu `grid[i][nj] != v` ⇒ hai ván cạnh nhau **ngược hướng** ⇒ chúng tạo chữ V ⇒ kẹt. (Gộp (b) và (d).) Vì sao luôn là chữ V mà không phải chữ Λ? Vì ô cạnh nằm **đúng phía ván đẩy tới**: với `v = 1` ô cạnh ở bên phải và là `/` ⇒ `\/` = V; với `v = −1` ô cạnh ở bên trái và là `\` ⇒ `\/` = V.
- Ngược lại (`grid[i][nj] == v`) hai ván song song tạo máng ⇒ bóng xuống hàng `i + 1` ở cột `nj`.

> **Quy tắc:** *Nếu* `0 <= nj < n` **và** `grid[i][nj] == grid[i][j]` *thì* bóng đi xuống cột `nj`; *ngược lại* bóng kẹt.

**Insight 2 — kết quả chỉ phụ thuộc vị trí hiện tại.** Bóng đang ở ô `(i, j)` sẽ ra ở đâu **không phụ thuộc** nó đã đến từ đâu. Gọi `exit(i, j)` là cột rơi ra khi bóng **vào ô `(i, j)` từ cạnh trên**, ta có quan hệ truy hồi:

- `exit(m, j) = j` (đã ra khỏi đáy),
- `exit(i, j) = -1` nếu kẹt ở hàng `i`,
- `exit(i, j) = exit(i + 1, j + grid[i][j])` nếu đi xuống được.

Từ đó ra **DFS** (đệ quy từ trên xuống) và **DP** (tính bảng từ đáy lên).

**Invariant khi mô phỏng:** trước khi xử lý hàng `i`, biến `j` là cột mà bóng đang ở **cạnh trên** của hàng `i`, và bóng chưa kẹt.

In [9]:
def step(grid, i, j):
    """Trả về cột ở hàng i+1 nếu bóng từ (i, j) đi xuống được, ngược lại -1."""
    n = len(grid[0])
    nj = j + grid[i][j]
    if nj < 0 or nj >= n or grid[i][nj] != grid[i][j]:
        return -1
    return nj

g = [[1, -1, 1, 1]]
for j in range(4):
    print(f"cột {j}: ván {g[0][j]:>2} -> {step(g, 0, j)}")

cột 0: ván  1 -> -1
cột 1: ván -1 -> -1
cột 2: ván  1 -> 3
cột 3: ván  1 -> -1


Giải thích kết quả trên với hàng `\ / \ \`:
- cột 0: `\`, ô phải là `/` ⇒ chữ V ⇒ `-1`.
- cột 1: `/`, ô trái là `\` ⇒ chữ V (cùng chữ V với cột 0!) ⇒ `-1`.
- cột 2: `\`, ô phải `\` ⇒ xuống cột 3.
- cột 3: `\`, không có ô phải ⇒ tường ⇒ `-1`.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Mô phỏng với quy tắc gọn `nj = j + grid[i][j]`

**Từng đoạn:**
1. Với mỗi cột xuất phát `start`, đặt `j = start`.
2. Lặp qua các hàng: tính `nj`; nếu vi phạm quy tắc thì đánh dấu `j = -1` và `break`.
3. Nếu không `break`, `j = nj` cho hàng tiếp theo.
4. Sau vòng lặp, `j` là đáp án (cột rơi ra hoặc `-1`).

Mẹo Python: dùng `for ... else` hoặc gán `j = -1` trước `break` để khỏi cần biến cờ `stuck`.

In [10]:
def find_ball_rule(grid):
    m, n = len(grid), len(grid[0])
    ans = []
    for start in range(n):
        j = start
        for i in range(m):
            nj = j + grid[i][j]
            if nj < 0 or nj >= n or grid[i][nj] != grid[i][j]:
                j = -1
                break
            j = nj
        ans.append(j)
    return ans

for g, want in examples:
    print(find_ball_rule(g), "| kỳ vọng", want)

[1, -1, -1, -1, -1] | kỳ vọng [1, -1, -1, -1, -1]
[-1] | kỳ vọng [-1]
[0, 1, 2, 3, 4, -1] | kỳ vọng [0, 1, 2, 3, 4, -1]


Code nộp LeetCode của Cách 2:

In [11]:
class Solution:
    def findBall(self, grid: List[List[int]]) -> List[int]:
        m, n = len(grid), len(grid[0])
        ans = []
        for start in range(n):
            j = start
            for i in range(m):
                nj = j + grid[i][j]
                if nj < 0 or nj >= n or grid[i][nj] != grid[i][j]:
                    j = -1
                    break
                j = nj
            ans.append(j)
        return ans

print(Solution().findBall(examples[0][0]))

[1, -1, -1, -1, -1]


**Độ phức tạp Cách 2:** Time `O(m·n)`, Space `O(1)` phụ (ngoài đáp án).

### 7.2 Cách 3 — DFS đệ quy theo quan hệ `exit(i, j)`

**Từng đoạn:**
1. `dfs(i, j)` trả về cột rơi ra khi bóng ở cạnh trên ô `(i, j)`.
2. **Điều kiện dừng:** `i == m` ⇒ bóng đã qua hàng cuối ⇒ trả `j`.
3. **Kẹt:** áp dụng quy tắc ⇒ trả `-1`.
4. **Đệ quy:** trả `dfs(i + 1, nj)`.

Phong cách này viết gần như nguyên văn công thức toán. Độ sâu đệ quy ≤ `m + 1 ≤ 101` nên an toàn.

In [12]:
def find_ball_dfs(grid):
    m, n = len(grid), len(grid[0])

    def dfs(i, j):
        if i == m:
            return j
        nj = j + grid[i][j]
        if nj < 0 or nj >= n or grid[i][nj] != grid[i][j]:
            return -1
        return dfs(i + 1, nj)

    return [dfs(0, j) for j in range(n)]

for g, want in examples:
    print(find_ball_dfs(g), "| kỳ vọng", want)

[1, -1, -1, -1, -1] | kỳ vọng [1, -1, -1, -1, -1]
[-1] | kỳ vọng [-1]
[0, 1, 2, 3, 4, -1] | kỳ vọng [0, 1, 2, 3, 4, -1]


Code nộp LeetCode của Cách 3:

In [13]:
class SolutionDFS:
    def findBall(self, grid: List[List[int]]) -> List[int]:
        m, n = len(grid), len(grid[0])

        def dfs(i: int, j: int) -> int:
            if i == m:
                return j
            nj = j + grid[i][j]
            if nj < 0 or nj >= n or grid[i][nj] != grid[i][j]:
                return -1
            return dfs(i + 1, nj)

        return [dfs(0, j) for j in range(n)]

print(SolutionDFS().findBall(examples[2][0]))

[0, 1, 2, 3, 4, -1]


**Độ phức tạp Cách 3:** Time `O(m·n)`; Space `O(m)` cho ngăn xếp đệ quy.

### 7.3 Cách 4 — DP từ đáy lên: bảng `exit` cho **mọi ô**

**Ý tưởng:** thay vì mô phỏng từng bóng, tính `exit[i][j]` cho **mọi** ô theo thứ tự hàng `m − 1 → 0`. Vì `exit[i][j]` chỉ phụ thuộc hàng `i + 1`, ta chỉ cần giữ **một mảng** `below` (hàng dưới) và cập nhật thành `cur` (hàng hiện tại).

**Từng đoạn:**
1. `below = [0, 1, ..., n − 1]` (bóng ở dưới đáy cột `j` thì "rơi ra" ở cột `j`).
2. Với hàng `i` từ dưới lên: `cur[j] = below[nj]` nếu đi được, ngược lại `-1`.
3. Sau hàng 0, `below` chính là đáp án.

**Khi nào hữu ích?** Khi có **nhiều truy vấn**: "thả bóng ở ô `(r, c)` bất kỳ thì ra đâu?" — giữ cả bảng `m × n` thì mỗi truy vấn trả lời `O(1)`. Với bài gốc thì độ phức tạp như cũ nhưng code **không có vòng lặp lồng theo từng bóng**.

In [14]:
def find_ball_dp(grid):
    m, n = len(grid), len(grid[0])
    below = list(range(n))               # hàng "ảo" m: rơi ra tại chính cột j
    for i in range(m - 1, -1, -1):
        cur = [-1] * n
        row = grid[i]
        for j in range(n):
            nj = j + row[j]
            if 0 <= nj < n and row[nj] == row[j]:
                cur[j] = below[nj]
        below = cur
    return below

for g, want in examples:
    print(find_ball_dp(g), "| kỳ vọng", want)

[1, -1, -1, -1, -1] | kỳ vọng [1, -1, -1, -1, -1]
[-1] | kỳ vọng [-1]
[0, 1, 2, 3, 4, -1] | kỳ vọng [0, 1, 2, 3, 4, -1]


Bản giữ **cả bảng** để trả lời truy vấn từ ô bất kỳ (minh họa ý "nhiều truy vấn"):

In [15]:
def exit_table(grid):
    m, n = len(grid), len(grid[0])
    table = [[-1] * n for _ in range(m + 1)]
    table[m] = list(range(n))
    for i in range(m - 1, -1, -1):
        for j in range(n):
            nj = j + grid[i][j]
            if 0 <= nj < n and grid[i][nj] == grid[i][j]:
                table[i][j] = table[i + 1][nj]
    return table

T = exit_table(examples[0][0])
for i, row in enumerate(T):
    print(f"hàng {i}: {row}")
print("thả bóng ở (3, 2) -> rơi ra cột", T[3][2])

hàng 0: [1, -1, -1, -1, -1]
hàng 1: [0, 1, -1, -1, -1]
hàng 2: [-1, 0, 1, -1, -1]
hàng 3: [0, 1, 2, -1, -1]
hàng 4: [-1, 0, 1, 2, 3]
hàng 5: [0, 1, 2, 3, 4]
thả bóng ở (3, 2) -> rơi ra cột 2


Code nộp LeetCode của Cách 4:

In [16]:
class SolutionDP:
    def findBall(self, grid: List[List[int]]) -> List[int]:
        m, n = len(grid), len(grid[0])
        below = list(range(n))
        for i in range(m - 1, -1, -1):
            row = grid[i]
            cur = [-1] * n
            for j in range(n):
                nj = j + row[j]
                if 0 <= nj < n and row[nj] == row[j]:
                    cur[j] = below[nj]
            below = cur
        return below

print(SolutionDP().findBall(examples[0][0]))

[1, -1, -1, -1, -1]


**Độ phức tạp Cách 4:** Time `O(m·n)` (mỗi ô xét đúng 1 lần), Space `O(n)` (bản 2 hàng) hoặc `O(m·n)` (bản bảng đầy đủ).

### 7.4 Cách 5 — Thả **tất cả** bóng cùng lúc, đi từng hàng

Đây là góc nhìn "song song": giữ mảng `pos` với `pos[b]` là cột hiện tại của bóng `b` (hoặc `-1` nếu đã kẹt). Mỗi hàng cập nhật tất cả bóng còn sống. Kết quả giống hệt, nhưng cách này giúp quan sát **cả đàn bóng** qua từng hàng — rất tiện để vẽ/trace.

In [17]:
def find_ball_parallel(grid, verbose=False):
    m, n = len(grid), len(grid[0])
    pos = list(range(n))
    for i in range(m):
        row = grid[i]
        for b in range(n):
            j = pos[b]
            if j == -1:
                continue
            nj = j + row[j]
            pos[b] = nj if 0 <= nj < n and row[nj] == row[j] else -1
        if verbose:
            print(f"sau hàng {i}: {pos}")
    return pos

print(find_ball_parallel(examples[0][0], verbose=True))

sau hàng 0: [1, 2, -1, -1, 3]
sau hàng 1: [2, -1, -1, -1, -1]
sau hàng 2: [1, -1, -1, -1, -1]
sau hàng 3: [2, -1, -1, -1, -1]
sau hàng 4: [1, -1, -1, -1, -1]
[1, -1, -1, -1, -1]


**Độ phức tạp Cách 5:** Time `O(m·n)`, Space `O(n)`.

## 8. TRACE từng bước

Ta trace **Ví dụ 1** cho từng quả bóng. Bảng trạng thái cho bóng 0 và bóng 1:

| Bóng | Hàng `i` | `j` | `grid[i][j]` | `nj` | `grid[i][nj]` | Quyết định |
|---|---|---|---|---|---|---|
| 0 | 0 | 0 | 1 | 1 | 1 | xuống cột 1 |
| 0 | 1 | 1 | 1 | 2 | 1 | xuống cột 2 |
| 0 | 2 | 2 | −1 | 1 | −1 | xuống cột 1 |
| 0 | 3 | 1 | 1 | 2 | 1 | xuống cột 2 |
| 0 | 4 | 2 | −1 | 1 | −1 | xuống cột 1 ⇒ **ra cột 1** |
| 1 | 0 | 1 | 1 | 2 | 1 | xuống cột 2 |
| 1 | 1 | 2 | 1 | 3 | −1 | **chữ V ⇒ kẹt** |

Hàm `trace_ball` in đúng các cột của bảng trên cho mọi quả bóng.

In [18]:
def trace_ball(grid, start):
    m, n = len(grid), len(grid[0])
    j = start
    print(f"--- bóng thả ở cột {start} ---")
    for i in range(m):
        v = grid[i][j]
        nj = j + v
        if nj < 0 or nj >= n:
            print(f"  i={i} j={j} ván={v:>2} nj={nj:>2} -> ngoài lưới: TƯỜNG, kẹt")
            return -1
        if grid[i][nj] != v:
            print(f"  i={i} j={j} ván={v:>2} nj={nj} grid[i][nj]={grid[i][nj]:>2} -> chữ V, kẹt")
            return -1
        print(f"  i={i} j={j} ván={v:>2} nj={nj} grid[i][nj]={grid[i][nj]:>2} -> xuống cột {nj}")
        j = nj
    print(f"  ra khỏi đáy ở cột {j}")
    return j

g1 = examples[0][0]
res = [trace_ball(g1, s) for s in range(len(g1[0]))]
print("Kết quả:", res)

--- bóng thả ở cột 0 ---
  i=0 j=0 ván= 1 nj=1 grid[i][nj]= 1 -> xuống cột 1
  i=1 j=1 ván= 1 nj=2 grid[i][nj]= 1 -> xuống cột 2
  i=2 j=2 ván=-1 nj=1 grid[i][nj]=-1 -> xuống cột 1
  i=3 j=1 ván= 1 nj=2 grid[i][nj]= 1 -> xuống cột 2
  i=4 j=2 ván=-1 nj=1 grid[i][nj]=-1 -> xuống cột 1
  ra khỏi đáy ở cột 1
--- bóng thả ở cột 1 ---
  i=0 j=1 ván= 1 nj=2 grid[i][nj]= 1 -> xuống cột 2
  i=1 j=2 ván= 1 nj=3 grid[i][nj]=-1 -> chữ V, kẹt
--- bóng thả ở cột 2 ---
  i=0 j=2 ván= 1 nj=3 grid[i][nj]=-1 -> chữ V, kẹt
--- bóng thả ở cột 3 ---
  i=0 j=3 ván=-1 nj=2 grid[i][nj]= 1 -> chữ V, kẹt
--- bóng thả ở cột 4 ---
  i=0 j=4 ván=-1 nj=3 grid[i][nj]=-1 -> xuống cột 3
  i=1 j=3 ván=-1 nj=2 grid[i][nj]= 1 -> chữ V, kẹt
Kết quả: [1, -1, -1, -1, -1]


Trace Cách 4 (DP từ đáy lên): in mảng `below` sau mỗi hàng. Đọc từ dưới lên: hàng cuối là `[-1, 0, 1, 2, 3]` nghĩa là bóng vào hàng 4 ở cột 1 sẽ ra cột 0, v.v.

In [19]:
def trace_dp(grid):
    m, n = len(grid), len(grid[0])
    below = list(range(n))
    print(f"khởi tạo (hàng ảo {m}): {below}")
    for i in range(m - 1, -1, -1):
        row = grid[i]
        cur = [-1] * n
        for j in range(n):
            nj = j + row[j]
            if 0 <= nj < n and row[nj] == row[j]:
                cur[j] = below[nj]
        print(f"hàng {i}: ván={row} -> exit={cur}")
        below = cur
    return below

print("Đáp án:", trace_dp(g1))

khởi tạo (hàng ảo 5): [0, 1, 2, 3, 4]
hàng 4: ván=[-1, -1, -1, -1, -1] -> exit=[-1, 0, 1, 2, 3]
hàng 3: ván=[1, 1, 1, 1, -1] -> exit=[0, 1, 2, -1, -1]
hàng 2: ván=[-1, -1, -1, 1, 1] -> exit=[-1, 0, 1, -1, -1]
hàng 1: ván=[1, 1, 1, -1, -1] -> exit=[0, 1, -1, -1, -1]
hàng 0: ván=[1, 1, 1, -1, -1] -> exit=[1, -1, -1, -1, -1]
Đáp án: [1, -1, -1, -1, -1]


## 9. Code nộp LeetCode

Bản nộp chính: mô phỏng với quy tắc gọn (Cách 2) — ngắn, dễ đọc, `O(1)` bộ nhớ phụ.

In [20]:
class Solution:
    def findBall(self, grid: List[List[int]]) -> List[int]:
        m, n = len(grid), len(grid[0])
        ans = []
        for start in range(n):
            j = start
            for i in range(m):
                nj = j + grid[i][j]          # cột mà ván đẩy bóng tới
                if nj < 0 or nj >= n or grid[i][nj] != grid[i][j]:
                    j = -1                   # đụng tường hoặc chữ V
                    break
                j = nj                       # đi xuống hàng i + 1 ở cột nj
            ans.append(j)
        return ans

sol = Solution()
for g, want in examples:
    assert sol.findBall(g) == want
print("Solution qua cả 3 ví dụ của đề")

Solution qua cả 3 ví dụ của đề


## 10. Vì sao thuật toán đúng

**Bổ đề (hành vi trong một hàng).** Giả sử bóng vào hàng `i` từ cạnh trên của ô `(i, j)`, ván `v = grid[i][j]`.

- Ván nghiêng về phía `j + v`, bóng lăn xuống mép ván và chạm vào **vách ngăn chung** giữa ô `j` và ô `nj = j + v` ở đúng góc dưới của ván.
- Nếu `nj` ngoài lưới: vách ngăn đó là **tường hộp** ⇒ bóng không đi đâu được ⇒ kẹt.
- Nếu `grid[i][nj] = v`: ván ô `nj` song song với ván ô `j`, đầu trên của nó nằm **cao hơn** chỗ bóng chạm vách, nên bóng rơi vào nửa dưới của ô `nj`, trượt theo ván xuống cạnh đáy của ô `(i, nj)` ⇒ sang hàng `i + 1` ở cột `nj`.
- Nếu `grid[i][nj] = −v`: ván ô `nj` dốc **về phía** ô `j`; hai ván gặp nhau ở đáy vách ngăn tạo chữ V ⇒ bóng nằm ở điểm thấp nhất, kẹt.

Bổ đề cho thấy 3 khả năng **loại trừ nhau và phủ hết** mọi trường hợp, và mỗi trường hợp được code xử lý đúng.

**Invariant.** Trước khi xử lý hàng `i`, `j` là cột bóng đang ở cạnh trên hàng `i` (và bóng chưa kẹt). Khởi đầu đúng (`i = 0, j = start`). Bổ đề đảm bảo sau mỗi bước invariant giữ cho hàng `i + 1` hoặc bóng kẹt (ta dừng, trả `-1`). Sau `m` hàng, `j` là cột rơi ra.

**Cách 4 (DP).** Vì vị trí tương lai chỉ phụ thuộc trạng thái hiện tại `(i, j)` (không phụ thuộc lịch sử), `exit(i, j) = exit(i + 1, nj)` khi đi được — quy nạp ngược theo `i` từ `m` xuống `0` chứng minh mọi giá trị trong bảng đúng.

## 11. Phân tích Time / Space Complexity

Gọi `m` = số hàng, `n` = số cột (`m, n ≤ 100`).

| Cách | Time | Space phụ | Ghi chú |
|---|---|---|---|
| 1. 4 trường hợp | `O(m·n)` | `O(1)` | mỗi bóng ≤ `m` bước |
| 2. Quy tắc gọn | `O(m·n)` | `O(1)` | worst case: mọi bóng đi hết `m` hàng |
| 3. DFS | `O(m·n)` | `O(m)` | stack đệ quy |
| 4. DP đáy lên | `O(m·n)` | `O(n)` / `O(m·n)` | mỗi ô xử lý 1 lần, trả lời truy vấn `O(1)` |
| 5. Song song | `O(m·n)` | `O(n)` | tiện trace |

**Best case** của cách mô phỏng: bóng kẹt ngay hàng 0 ⇒ `O(n)` tổng. **Worst case**: lưới kiểu ví dụ 3 (`\`/`/` xen kẽ theo hàng) — gần như mọi bóng đi hết `m` hàng ⇒ `m·n = 10⁴` bước, tức chạy tức thì. DP luôn `Θ(m·n)` vì tính mọi ô kể cả ô không bóng nào đi qua.

Một nhận xét thú vị: **không có hai bóng nào ở cùng một ô** tại cùng một hàng (xem bài tập 5), nên tổng số bước của tất cả bóng ≤ `m·n` ngay cả khi tính theo "song song".

In [21]:
# Đo nhanh trên lưới lớn nhất 100 x 100, dạng worst case (xen kẽ hàng 1 / -1)
big = [[1] * 100 if i % 2 == 0 else [-1] * 100 for i in range(100)]
for name, f in [("rule", find_ball_rule), ("dfs", find_ball_dfs), ("dp", find_ball_dp), ("parallel", find_ball_parallel)]:
    t = time.perf_counter()
    r = f(big)
    print(f"{name:>8}: {time.perf_counter() - t:.4f}s, số bóng rơi ra = {sum(x != -1 for x in r)}")

    rule: 0.0007s, số bóng rơi ra = 99
     dfs: 0.0011s, số bóng rơi ra = 99
      dp: 0.0006s, số bóng rơi ra = 99
parallel: 0.0008s, số bóng rơi ra = 99


## 12. Bộ test

### 12.1 Mô hình kiểm chứng độc lập: chia ô thành hai tam giác

Để không "tự kiểm tra mình bằng chính mình", ta viết một mô phỏng dựa trên **hình học** chứ không dựa trên quy tắc `nj`:

- Ván `\` chia ô thành tam giác **trên-phải** (cạnh `top` + `right`) và tam giác **dưới-trái** (cạnh `left` + `bottom`).
- Ván `/` chia ô thành tam giác **trên-trái** (`top` + `left`) và **dưới-phải** (`right` + `bottom`).

Bóng vào một ô qua một cạnh, ở lại trong tam giác chứa cạnh đó, và rời tam giác qua **cạnh còn lại**. Nếu cạnh còn lại là `top` thì bóng phải đi **lên** — không thể vì trọng lực ⇒ kẹt (đó chính là chữ V). Nếu cạnh còn lại là `left`/`right` ở biên ⇒ tường ⇒ kẹt.

In [22]:
def triangle_model(grid):
    m, n = len(grid), len(grid[0])
    TRI = {
        1:  [{"top", "right"}, {"left", "bottom"}],   # ván "\"
        -1: [{"top", "left"}, {"right", "bottom"}],   # ván "/"
    }
    res = []
    for start in range(n):
        i, j, entry = 0, start, "top"
        out = None
        while out is None:
            if i == m:
                out = j; break
            tri = next(t for t in TRI[grid[i][j]] if entry in t)
            (leave,) = tri - {entry}
            if leave == "bottom":
                i, entry = i + 1, "top"
            elif leave == "top":
                out = -1                               # phải đi lên => kẹt (chữ V)
            elif leave == "right":
                if j == n - 1: out = -1
                else: j, entry = j + 1, "left"
            else:  # left
                if j == 0: out = -1
                else: j, entry = j - 1, "right"
        res.append(out)
    return res

for g, want in examples:
    print(triangle_model(g), "| kỳ vọng", want)

[1, -1, -1, -1, -1] | kỳ vọng [1, -1, -1, -1, -1]
[-1] | kỳ vọng [-1]
[0, 1, 2, 3, 4, -1] | kỳ vọng [0, 1, 2, 3, 4, -1]


### 12.2 Ví dụ của đề + edge case

Các edge case:
- `1 × 1`: `[[1]]` (tường phải), `[[-1]]` (tường trái).
- Một hàng `\ \` → bóng 0 ra cột 1, bóng 1 kẹt tường.
- Một cột duy nhất nhiều hàng: luôn kẹt ngay hàng đầu (mọi ván đều đẩy vào tường).
- Toàn `1`: chỉ bóng ở các cột mà sau `m` bước vẫn chưa chạm tường mới ra (bóng `j` ra cột `j + m` nếu `j + m ≤ n − 1`).

In [23]:
impls = {
    "brute": find_ball_brute,
    "rule": find_ball_rule,
    "dfs": find_ball_dfs,
    "dp": find_ball_dp,
    "parallel": find_ball_parallel,
    "Solution": Solution().findBall,
    "SolutionDP": SolutionDP().findBall,
    "triangle": triangle_model,
}

edge = [
    ([[1]], [-1]),
    ([[-1]], [-1]),
    ([[1, 1]], [1, -1]),
    ([[-1, -1]], [-1, 0]),
    ([[1, -1]], [-1, -1]),                  # một chữ V duy nhất
    ([[-1, 1]], [-1, -1]),                  # hai bóng đều bị đẩy vào tường
    ([[1], [1], [-1]], [-1]),               # một cột
    ([[1, 1, 1, 1]] * 2, [2, 3, -1, -1]),   # toàn "\": bóng j ra j + m
]
for g, want in examples + edge:
    for name, f in impls.items():
        assert f(g) == want, (name, g, f(g), want)
print("Tất cả cách qua", len(examples) + len(edge), "test cố định")

Tất cả cách qua 11 test cố định


### 12.3 Ngẫu nhiên có seed cố định (đúng ràng buộc đề)

Sinh 600 lưới ngẫu nhiên kích thước `1..8 × 1..8` (dễ có đủ kiểu kẹt) và 20 lưới `100 × 100` (kích thước tối đa), so sánh mọi cách với mô hình tam giác.

In [24]:
rng = random.Random(1706)
cnt = 0
for _ in range(600):
    m, n = rng.randint(1, 8), rng.randint(1, 8)
    g = [[rng.choice((1, -1)) for _ in range(n)] for _ in range(m)]
    ref = triangle_model(g)
    for name, f in impls.items():
        assert f(g) == ref, (name, g)
    cnt += 1
for _ in range(20):
    # lưới lớn có thiên hướng để nhiều bóng đi xa: hàng gần như đồng nhất
    g = []
    for i in range(100):
        base = rng.choice((1, -1))
        g.append([base if rng.random() < 0.97 else -base for _ in range(100)])
    ref = triangle_model(g)
    for name, f in impls.items():
        assert f(g) == ref, name
    cnt += 1
print("OK", cnt, "lưới ngẫu nhiên, mọi cách khớp mô hình tam giác")

OK 620 lưới ngẫu nhiên, mọi cách khớp mô hình tam giác


### 12.4 Kiểm tra tính chất: không hai bóng nào rơi ra cùng một cột

Đây là một **tính chất** (property) của bài, dùng làm kiểm tra phụ: các giá trị khác `-1` trong đáp án luôn **phân biệt**.

In [25]:
rng = random.Random(17)
for _ in range(300):
    m, n = rng.randint(1, 10), rng.randint(1, 10)
    g = [[rng.choice((1, -1)) for _ in range(n)] for _ in range(m)]
    out = [x for x in Solution().findBall(g) if x != -1]
    assert len(out) == len(set(out))
print("Tính chất 'không trùng cột ra' đúng trên 300 lưới")

Tính chất 'không trùng cột ra' đúng trên 300 lưới


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Quên kiểm tra tường trái (Python không báo lỗi với chỉ số âm)

In [26]:
def wrong_no_left_wall(grid):
    m, n = len(grid), len(grid[0])
    ans = []
    for start in range(n):
        j = start
        for i in range(m):
            v = grid[i][j]
            nj = j + v
            if nj >= n or grid[i][nj] != v:   # SAI: thiếu nj < 0
                j = -1; break
            j = nj
        ans.append(j)
    return ans

g = [[-1, -1], [1, 1]]
print("sai :", wrong_no_left_wall(g))
print("đúng:", find_ball_rule(g))

sai : [0, 1]
đúng: [-1, 1]


**Vì sao sai:** bóng 0 ở cột 0 gặp ván `/`, `nj = -1`. Thay vì báo "tường", code đọc `grid[0][-1]` = ô **cột cuối** (cũng là `-1`) nên tưởng đi được, gán `j = -1`. Ở hàng sau, `grid[1][-1]` lại là cột cuối, bóng "đi tiếp" và kết thúc ở cột 0 — đáp án sai `0` thay vì `-1`. **Sửa:** luôn kiểm tra `nj < 0 or nj >= n` **trước** khi đọc `grid[i][nj]`.

### Lỗi 2 — Chỉ kiểm tra tường, quên chữ V

In [27]:
def wrong_no_v(grid):
    m, n = len(grid), len(grid[0])
    ans = []
    for start in range(n):
        j = start
        for i in range(m):
            nj = j + grid[i][j]
            if nj < 0 or nj >= n:              # SAI: thiếu điều kiện chữ V
                j = -1; break
            j = nj
        ans.append(j)
    return ans

g = [[1, -1]]
print("sai :", wrong_no_v(g))
print("đúng:", find_ball_rule(g))

sai : [1, 0]
đúng: [-1, -1]


**Vì sao sai:** lưới `\ /` là một chữ V; bóng 0 lăn sang phải chạm ván `/` và kẹt, bóng 1 lăn sang trái chạm ván `\` và kẹt. Bản sai để chúng "đổi chỗ" cho nhau ra `[1, 0]`. **Sửa:** thêm `grid[i][nj] != grid[i][j]`.

### Lỗi 3 — So ô cạnh ở **hàng dưới** thay vì cùng hàng

In [28]:
def wrong_next_row(grid):
    m, n = len(grid), len(grid[0])
    ans = []
    for start in range(n):
        j = start
        for i in range(m):
            nj = j + grid[i][j]
            if nj < 0 or nj >= n:
                j = -1; break
            if i + 1 < m and grid[i + 1][nj] != grid[i][j]:   # SAI: xem hàng i+1
                j = -1; break
            j = nj
        ans.append(j)
    return ans

g = [[1, -1, -1], [1, 1, 1]]
print("sai :", wrong_next_row(g))
print("đúng:", find_ball_rule(g))

sai : [2, -1, -1]
đúng: [-1, -1, 2]


**Vì sao sai:** chữ V được tạo bởi hai ván **cạnh nhau trong cùng một hàng**. Ở ví dụ, hàng 0 là `\ / /` nên bóng 0 kẹt ngay ở chữ V cột 0–1; bản sai lại nhìn xuống hàng 1 (`\ \ \`), thấy "cùng hướng" và cho bóng đi tiếp. **Sửa:** so `grid[i][nj]` (cùng hàng `i`).

### Lỗi 4 — Kẹt rồi nhưng vẫn ghi cột hiện tại

In [29]:
def wrong_append_j(grid):
    m, n = len(grid), len(grid[0])
    ans = []
    for start in range(n):
        j = start
        for i in range(m):
            nj = j + grid[i][j]
            if nj < 0 or nj >= n or grid[i][nj] != grid[i][j]:
                break                      # SAI: quên đặt j = -1
            j = nj
        ans.append(j)
    return ans

g = examples[0][0]
print("sai :", wrong_append_j(g))
print("đúng:", find_ball_rule(g))

sai : [1, 2, 2, 3, 3]
đúng: [1, -1, -1, -1, -1]


**Vì sao sai:** `break` chỉ thoát vòng lặp; `j` vẫn giữ cột lúc kẹt nên đáp án ghi một cột "ảo" thay vì `-1`. **Sửa:** đặt `j = -1` trước `break`, hoặc dùng cờ `stuck`, hoặc cấu trúc `for ... else`.

### Lỗi 5 — Nhầm chiều ván (`1` là `/`)

In [30]:
def wrong_direction(grid):
    m, n = len(grid), len(grid[0])
    ans = []
    for start in range(n):
        j = start
        for i in range(m):
            nj = j - grid[i][j]                # SAI: đảo hướng
            if nj < 0 or nj >= n or grid[i][nj] != grid[i][j]:
                j = -1; break
            j = nj
        ans.append(j)
    return ans

g = examples[2][0]
print("sai :", wrong_direction(g))
print("đúng:", find_ball_rule(g))

sai : [-1, 1, 2, 3, 4, 5]
đúng: [0, 1, 2, 3, 4, -1]


**Vì sao sai:** đề nói `1` đẩy bóng sang **phải**. Đảo dấu thì kết quả là ảnh gương sai lệch hoàn toàn. **Sửa:** `nj = j + grid[i][j]`. Mẹo nhớ: "1 = dương = tăng chỉ số = sang phải".

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **Đề mô phỏng trên lưới nhỏ** (`≤ 100`) ⇒ cứ mô phỏng trực tiếp, đừng tìm công thức phức tạp. Thời gian nên dành cho việc **mã hóa điều kiện đúng**.
- **Giá trị ô = hướng** (`±1`) ⇒ dùng ngay làm độ dời `j + v`, bỏ được `if`.
- **"Hai ô cạnh nhau ngược hướng"** ⇒ so sánh `grid[i][nj] != grid[i][j]` thay vì liệt kê từng cặp.
- **Trạng thái tương lai chỉ phụ thuộc vị trí hiện tại** ⇒ có thể DP/ghi nhớ (`exit[i][j]`) để trả lời nhiều truy vấn.
- Trong Python, **luôn kiểm tra biên trước khi đọc chỉ số** vì chỉ số âm hợp lệ.
- Viết hàm `trace` in từng bước giúp gỡ lỗi mô phỏng nhanh hơn nhiều so với đoán.

## 15. Biến thể của bài

1. **Truy vấn nhiều ô xuất phát** (thả bóng từ ô `(r, c)` bất kỳ): dùng bảng `exit_table` — `O(m·n)` tiền xử lý, `O(1)` mỗi truy vấn.
2. **Đếm số bóng rơi ra** hoặc **tổng cột rơi ra**: chỉ cần đáp án gốc rồi thống kê.
3. **Đảo ngược**: biết cột đáy, tìm cột xuất phát (bài tập 5) — dùng tính chất đơn ánh.
4. **Lật ván**: được lật tối đa một ô, hỏi có làm bóng `k` rơi ra không — mô phỏng với mỗi ô lật (`O(m·n)` lựa chọn × `O(m)` mô phỏng).
5. **Lưới có thêm ô trống / ô chặn** (giống LC 1391 Check if There is a Valid Path in a Grid): mở rộng bảng chuyển trạng thái theo loại ô.
6. Liên quan: **LC 1559, LC 1391** (đi trên lưới theo kiểu ô), **LC 2101** (mô phỏng lan truyền), **LC 489** (mô phỏng robot).

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mô phỏng 4 trường hợp | dịch nguyên văn đề | `O(m·n)` | `O(1)` | lần đầu đọc đề, làm baseline |
| 2. Quy tắc gọn `nj` | gộp tường + chữ V | `O(m·n)` | `O(1)` | **nộp bài**, phỏng vấn |
| 3. DFS đệ quy | `exit(i,j) = exit(i+1, nj)` | `O(m·n)` | `O(m)` | khi thích viết theo công thức |
| 4. DP đáy lên | tính `exit` cho mọi ô | `O(m·n)` | `O(n)` / `O(m·n)` | nhiều truy vấn từ ô bất kỳ |
| 5. Song song | cập nhật cả đàn bóng mỗi hàng | `O(m·n)` | `O(n)` | trace / trực quan hóa |
| Mô hình tam giác | hình học thuần | `O(m·n)` | `O(1)` | kiểm chứng độc lập trong test |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** Không chạy code, đoán `findBall` cho:
(a) `[[1, 1], [-1, -1]]`; (b) `[[-1, 1, -1]]`; (c) `[[1, 1, 1], [1, 1, 1]]`.

**Bài 2 (tính tay):** Với Ví dụ 1, bóng thả ở cột 4 kẹt ở **hàng nào**, vì lý do gì (tường hay chữ V, giữa những cột nào)?

**Bài 3 (điền TODO):** Hoàn thiện `ball_path(grid, start)` trả về **danh sách các cột** bóng đi qua ở cạnh trên mỗi hàng (kể cả cột rơi ra ở cuối), hoặc `None` nếu bóng kẹt.

**Bài 4 (tự viết lại):** Không nhìn code, viết lại Cách 4 (DP từ đáy lên, chỉ giữ một mảng `below`) và kiểm tra trên 3 ví dụ của đề.

**Bài 5 (mini challenge):** Viết `bottom_to_start(grid)` trả về dict `{cột_đáy: cột_xuất_phát}` cho mọi bóng rơi ra. Chứng minh (bằng lời) và kiểm tra (bằng assert trên lưới ngẫu nhiên) rằng không có hai bóng rơi ra cùng một cột.

In [31]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def ball_path_todo(grid, start):
    m, n = len(grid), len(grid[0])
    path = [start]
    j = start
    for i in range(m):
        # TODO: tính nj, nếu kẹt trả None
        # TODO: cập nhật j và thêm vào path
        pass
    return path

In [32]:
# Bài 4 — TODO (chỉ định nghĩa, không gọi)
def find_ball_dp_todo(grid):
    m, n = len(grid), len(grid[0])
    below = list(range(n))
    # TODO: duyệt hàng từ m-1 về 0, tính cur từ below
    return below

In [33]:
# Bài 5 — TODO (chỉ định nghĩa, không gọi)
def bottom_to_start_todo(grid):
    # TODO: dùng một lời giải bất kỳ, đảo ánh xạ start -> bottom
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: áp dụng quy tắc `nj` cho từng bóng. Mức 2: ở (a), bóng 1 đụng tường phải ngay hàng 0. Mức 3: ở (c), toàn `\`: bóng `j` ra ở `j + 2` nếu `j + 2 ≤ 2`.

**Bài 2.** Mức 1: dùng `trace_ball(g1, 4)`. Mức 2: hàng 0 cột 4 là `/` đẩy sang trái, cột 3 cũng `/` ⇒ đi được. Mức 3: hàng 1 bóng ở cột 3, ván `/`, ô trái cột 2 là `\` ⇒ chữ V giữa cột 2–3.

**Bài 3.** Mức 1: giống Cách 2 nhưng nối `nj` vào list. Mức 2: khi kẹt `return None` ngay. Mức 3: độ dài path thành công luôn là `m + 1`.

**Bài 4.** Mức 1: hàng ảo `m` có `below[j] = j`. Mức 2: `cur[j] = below[nj]` khi đi được. Mức 3: nhớ tạo `cur` **mới** mỗi hàng, đừng sửa `below` tại chỗ (vì `below[nj]` có thể bị ghi đè trước khi đọc).

**Bài 5.** Mức 1: lấy `ans = findBall(grid)`, với mỗi `start` có `ans[start] != -1` thì `d[ans[start]] = start`. Mức 2: nếu hai bóng vào cùng ô `(i, j)` từ trái và phải thì ô `(i−1, j)` phải vừa là `\` vừa là `/`. Mức 3: quy nạp theo hàng — hai bóng khác cột không bao giờ nhập vào cùng cột.

## 19. Đáp án tham khảo cho bài tập

**Bài 1:** (a) `[0, -1]`: bóng 0: hàng 0 `\`, cột 1 `\` ⇒ cột 1; hàng 1 `/`, cột 0 `/` ⇒ cột 0. Bóng 1: tường phải. (b) `[-1, -1, -1]`: bóng 0 tường trái; bóng 1 `\` gặp `/` (chữ V cột 1–2); bóng 2 `/` gặp `\` (cùng chữ V). (c) `[2, -1, -1]`.

In [34]:
assert Solution().findBall([[1, 1], [-1, -1]]) == [0, -1]
assert Solution().findBall([[-1, 1, -1]]) == [-1, -1, -1]
assert Solution().findBall([[1, 1, 1], [1, 1, 1]]) == [2, -1, -1]
print("Bài 1 đúng")

Bài 1 đúng


**Bài 2:** bóng thả cột 4 qua hàng 0 (ván `/`, cột 3 cũng `/`) sang cột 3; ở hàng 1, ván `/` muốn sang cột 2 nhưng cột 2 là `\` ⇒ **chữ V giữa cột 2–3, hàng 1**. Khớp mô tả của đề.

In [35]:
r = trace_ball(g1, 4)
assert r == -1

--- bóng thả ở cột 4 ---
  i=0 j=4 ván=-1 nj=3 grid[i][nj]=-1 -> xuống cột 3
  i=1 j=3 ván=-1 nj=2 grid[i][nj]= 1 -> chữ V, kẹt


In [36]:
# Bài 3 — đáp án
def ball_path(grid, start):
    m, n = len(grid), len(grid[0])
    path = [start]
    j = start
    for i in range(m):
        nj = j + grid[i][j]
        if nj < 0 or nj >= n or grid[i][nj] != grid[i][j]:
            return None
        j = nj
        path.append(j)
    return path

print(ball_path(examples[2][0], 0))
assert ball_path(examples[2][0], 0) == [0, 1, 0, 1, 0]
assert ball_path(examples[2][0], 5) is None
assert ball_path(g1, 0) == [0, 1, 2, 1, 2, 1]
print("Bài 3 đúng")

[0, 1, 0, 1, 0]
Bài 3 đúng


In [37]:
# Bài 4 — đáp án (viết lại Cách 4)
def find_ball_dp_again(grid):
    m, n = len(grid), len(grid[0])
    below = list(range(n))
    for i in range(m - 1, -1, -1):
        cur = [-1] * n
        for j in range(n):
            nj = j + grid[i][j]
            if 0 <= nj < n and grid[i][nj] == grid[i][j]:
                cur[j] = below[nj]
        below = cur
    return below

for g, want in examples:
    assert find_ball_dp_again(g) == want
print("Bài 4 đúng")

Bài 4 đúng


In [38]:
# Bài 5 — đáp án
def bottom_to_start(grid):
    ans = Solution().findBall(grid)
    d = {}
    for start, bottom in enumerate(ans):
        if bottom != -1:
            assert bottom not in d          # đơn ánh
            d[bottom] = start
    return d

print(bottom_to_start(g1))
print(bottom_to_start(examples[2][0]))
rng = random.Random(5)
for _ in range(200):
    m, n = rng.randint(1, 9), rng.randint(1, 9)
    g = [[rng.choice((1, -1)) for _ in range(n)] for _ in range(m)]
    d = bottom_to_start(g)
    assert len(d) == sum(x != -1 for x in Solution().findBall(g))
print("Bài 5 đúng")

{1: 0}
{0: 0, 1: 1, 2: 2, 3: 3, 4: 4}
Bài 5 đúng


**Chứng minh bài 5 (bằng lời):** giả sử hai bóng cùng ở cạnh trên ô `(i, j)`. Một bóng phải đến từ `(i−1, j−1)` qua máng `\\` (nên `grid[i−1][j] = 1`), bóng kia từ `(i−1, j+1)` qua máng `//` (nên `grid[i−1][j] = −1`). Mâu thuẫn. Vậy hai bóng khác cột ở hàng `i − 1` thì vẫn khác cột ở hàng `i`; quy nạp tới đáy.

## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
# Template mô phỏng bóng trên lưới ván chéo
for start in range(n):
    j = start
    for i in range(m):
        nj = j + grid[i][j]                  # hướng = giá trị ô
        if nj < 0 or nj >= n or grid[i][nj] != grid[i][j]:
            j = -1; break                    # tường hoặc chữ V
        j = nj
    ans.append(j)

# Template DP từ đáy lên (nhiều truy vấn)
below = list(range(n))
for i in range(m - 1, -1, -1):
    cur = [-1] * n
    for j in range(n):
        nj = j + grid[i][j]
        if 0 <= nj < n and grid[i][nj] == grid[i][j]:
            cur[j] = below[nj]
    below = cur
```

**Checklist trước khi submit:**
- [ ] Kiểm tra `nj < 0` **và** `nj >= n` **trước** khi đọc `grid[i][nj]` (nhớ chỉ số âm trong Python).
- [ ] So ô cạnh **cùng hàng** `i`, không phải hàng `i + 1`.
- [ ] Khi kẹt, ghi `-1` (không ghi cột hiện tại).
- [ ] `1` = sang phải (`j + 1`), `-1` = sang trái.
- [ ] Thử `[[1]]`, `[[-1]]`, `[[1, -1]]`, và 3 ví dụ của đề.
- [ ] Trả về `List[int]` độ dài đúng `n`.